In [1]:
import ast

code = """
def add(a, b):
    return a + b
"""

tree = ast.parse(code)
print(ast.dump(tree, indent=2))

Module(
  body=[
    FunctionDef(
      name='add',
      args=arguments(
        args=[
          arg(arg='a'),
          arg(arg='b')]),
      body=[
        Return(
          value=BinOp(
            left=Name(id='a', ctx=Load()),
            op=Add(),
            right=Name(id='b', ctx=Load())))])])


In [5]:
import os
os.chdir("..")
print(os.getcwd())


e:\codebase-rag-assistant


In [6]:
print(os.listdir("target_repo"))

['.devcontainer', '.editorconfig', '.git', '.github', '.gitignore', '.pre-commit-config.yaml', '.readthedocs.yaml', 'CHANGES.rst', 'docs', 'examples', 'LICENSE.txt', 'pyproject.toml', 'README.md', 'src', 'tests', 'uv.lock']


In [7]:
print(os.listdir("target_repo/src"))

['flask']


In [8]:
flask_src = "target_repo/src/flask"
print(os.listdir(flask_src))

['app.py', 'blueprints.py', 'cli.py', 'config.py', 'ctx.py', 'debughelpers.py', 'globals.py', 'helpers.py', 'json', 'logging.py', 'py.typed', 'sansio', 'sessions.py', 'signals.py', 'templating.py', 'testing.py', 'typing.py', 'views.py', 'wrappers.py', '__init__.py', '__main__.py']


In [9]:
import ast

file_path = os.path.join(flask_src, "helpers.py")

with open(file_path, "r", encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)

for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        print(f"{type(node).__name__}: {node.name} (lines {node.lineno}-{node.end_lineno})")

FunctionDef: get_debug_flag (lines 28-33)
FunctionDef: get_load_dotenv (lines 36-48)
FunctionDef: stream_with_context (lines 52-54)
FunctionDef: stream_with_context (lines 58-60)
FunctionDef: stream_with_context (lines 63-148)
FunctionDef: make_response (lines 151-197)
FunctionDef: url_for (lines 200-251)
FunctionDef: redirect (lines 254-278)
FunctionDef: abort (lines 281-301)
FunctionDef: get_template_attribute (lines 304-323)
FunctionDef: flash (lines 326-357)
FunctionDef: get_flashed_messages (lines 360-399)
FunctionDef: _prepare_send_file_kwargs (lines 402-414)
FunctionDef: send_file (lines 417-540)
FunctionDef: send_from_directory (lines 543-584)
FunctionDef: get_root_path (lines 587-641)
FunctionDef: _split_blueprint_path (lines 645-651)
ClassDef: _CollectErrors (lines 654-682)
FunctionDef: generator (lines 126-141)
FunctionDef: __init__ (lines 659-660)
FunctionDef: __enter__ (lines 662-663)
FunctionDef: __exit__ (lines 665-674)
FunctionDef: raise_any (lines 676-682)
FunctionDef:

In [10]:
import json

with open("data/chunks.json") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
print(json.dumps(chunks[5], indent=2))  # print one example chunk

Total chunks: 442
{
  "type": "FunctionDef",
  "name": "wrapper",
  "code": "    def wrapper(self: Flask, *args: t.Any, **kwargs: t.Any) -> t.Any:\n        if not args:\n            args = (app_ctx._get_current_object(),)\n        elif not isinstance(args[0], AppContext):\n            args = (app_ctx._get_current_object(), *args)\n\n        return f(self, *args, **kwargs)",
  "file": "target_repo/src/flask\\app.py",
  "start_line": 99,
  "end_line": 105,
  "docstring": ""
}


In [11]:
from sentence_transformers import SentenceTransformer

# Using a general-purpose model to start — we'll swap to a code-specific one shortly
model = SentenceTransformer("all-MiniLM-L6-v2")

# Quick test on one chunk
sample_text = chunks[5]["code"]
embedding = model.encode(sample_text)
print(embedding.shape)
print(embedding[:5])  # peek at first 5 numbers

e:\codebase-rag-assistant\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\codebase-rag-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activ

(384,)
[-0.06731227  0.06183861 -0.02232258  0.0681266   0.0618494 ]


In [12]:
import chromadb

# Create a persistent ChromaDB client (saves to disk so we don't re-embed every time)
client = chromadb.PersistentClient(path="data/chroma_db")

# Create (or get) a collection to store our chunks
collection = client.get_or_create_collection(name="flask_codebase")

# Prepare data for insertion
documents = [chunk["code"] for chunk in chunks]
metadatas = [
    {
        "file": chunk["file"],
        "name": chunk["name"],
        "type": chunk["type"],
        "start_line": chunk["start_line"],
        "end_line": chunk["end_line"],
    }
    for chunk in chunks
]
ids = [f"chunk_{i}" for i in range(len(chunks))]

# Embed and add in batches (ChromaDB can be slow with huge single inserts)
batch_size = 50
for i in range(0, len(documents), batch_size):
    batch_docs = documents[i:i+batch_size]
    batch_embeds = model.encode(batch_docs).tolist()
    batch_meta = metadatas[i:i+batch_size]
    batch_ids = ids[i:i+batch_size]

    collection.add(
        documents=batch_docs,
        embeddings=batch_embeds,
        metadatas=batch_meta,
        ids=batch_ids
    )
    print(f"Inserted batch {i} to {i+len(batch_docs)}")

print("Done! Total chunks in collection:", collection.count())

Inserted batch 0 to 50
Inserted batch 50 to 100
Inserted batch 100 to 150
Inserted batch 150 to 200
Inserted batch 200 to 250
Inserted batch 250 to 300
Inserted batch 300 to 350
Inserted batch 350 to 400
Inserted batch 400 to 442
Done! Total chunks in collection: 442


In [13]:
query = "how does flask handle url routing"
query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"--- Result {i+1} ---")
    print(f"File: {meta['file']}, Function: {meta['name']} (lines {meta['start_line']}-{meta['end_line']})")
    print(doc[:200])
    print()

--- Result 1 ---
File: target_repo/src/flask\app.py, Function: url_for (lines 1105-1225)
    def url_for(
        self,
        /,
        endpoint: str,
        *,
        _anchor: str | None = None,
        _method: str | None = None,
        _scheme: str | None = None,
        _externa

--- Result 2 ---
File: target_repo/src/flask\debughelpers.py, Function: __init__ (lines 57-78)
    def __init__(self, request: Request) -> None:
        exc = request.routing_exception
        assert isinstance(exc, RequestRedirect)
        buf = [
            f"A request was sent to '{request.

--- Result 3 ---
File: target_repo/src/flask\helpers.py, Function: url_for (lines 200-251)
def url_for(
    endpoint: str,
    *,
    _anchor: str | None = None,
    _method: str | None = None,
    _scheme: str | None = None,
    _external: bool | None = None,
    **values: t.Any,
) -> str:

--- Result 4 ---
File: target_repo/src/flask\wrappers.py, Function: Request (lines 18-219)
class Request(RequestBase):


In [14]:
from rank_bm25 import BM25Okapi

# Tokenize each chunk's code (simple whitespace split is fine to start)
tokenized_corpus = [doc.split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

# Test with the same query as before
query = "how does flask handle url routing"
tokenized_query = query.split()

scores = bm25.get_scores(tokenized_query)

# Get top 5 indices by score
import numpy as np
top_indices = np.argsort(scores)[::-1][:5]

for idx in top_indices:
    meta = metadatas[idx]
    print(f"Score: {scores[idx]:.2f} | File: {meta['file']}, Function: {meta['name']} (lines {meta['start_line']}-{meta['end_line']})")

Score: 7.83 | File: target_repo/src/flask\sansio\scaffold.py, Function: errorhandler (lines 606-647)
Score: 6.69 | File: target_repo/src/flask\ctx.py, Function: has_request_context (lines 209-232)
Score: 6.65 | File: target_repo/src/flask\ctx.py, Function: has_app_context (lines 235-257)
Score: 6.58 | File: target_repo/src/flask\sansio\scaffold.py, Function: add_url_rule (lines 376-441)
Score: 6.35 | File: target_repo/src/flask\ctx.py, Function: match_request (lines 405-414)


In [15]:
# Dense ranking (from ChromaDB query)
dense_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)
dense_ids = dense_results["ids"][0]  # e.g. ['chunk_23', 'chunk_5', ...]

# Sparse ranking (from BM25)
sparse_top_indices = np.argsort(scores)[::-1][:10]
sparse_ids = [f"chunk_{idx}" for idx in sparse_top_indices]

print("Dense order:", dense_ids)
print("Sparse order:", sparse_ids)

Dense order: ['chunk_30', 'chunk_142', 'chunk_159', 'chunk_247', 'chunk_3', 'chunk_324', 'chunk_357', 'chunk_136', 'chunk_375', 'chunk_428']
Sparse order: ['chunk_436', 'chunk_107', 'chunk_108', 'chunk_428', 'chunk_128', 'chunk_136', 'chunk_68', 'chunk_142', 'chunk_14', 'chunk_208']


In [16]:
def reciprocal_rank_fusion(dense_ids, sparse_ids, k=60):
    scores = {}
    for rank, doc_id in enumerate(dense_ids):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    for rank, doc_id in enumerate(sparse_ids):
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

fused = reciprocal_rank_fusion(dense_ids, sparse_ids)

print("--- Hybrid (RRF) Top 5 ---")
for doc_id, score in fused[:5]:
    idx = int(doc_id.split("_")[1])
    meta = metadatas[idx]
    print(f"RRF Score: {score:.4f} | File: {meta['file']}, Function: {meta['name']} (lines {meta['start_line']}-{meta['end_line']})")

--- Hybrid (RRF) Top 5 ---
RRF Score: 0.0308 | File: target_repo/src/flask\debughelpers.py, Function: __init__ (lines 57-78)
RRF Score: 0.0299 | File: target_repo/src/flask\sansio\scaffold.py, Function: add_url_rule (lines 376-441)
RRF Score: 0.0299 | File: target_repo/src/flask\debughelpers.py, Function: FormDataRoutingRedirect (lines 50-78)
RRF Score: 0.0164 | File: target_repo/src/flask\app.py, Function: url_for (lines 1105-1225)
RRF Score: 0.0164 | File: target_repo/src/flask\sansio\scaffold.py, Function: errorhandler (lines 606-647)


In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

e:\codebase-rag-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4495.41it/s]


In [18]:
# Take top 10 fused candidates
candidate_ids = [doc_id for doc_id, _ in fused[:10]]
candidate_texts = [documents[int(doc_id.split("_")[1])] for doc_id in candidate_ids]

# Cross-encoder expects [query, passage] pairs
pairs = [[query, text] for text in candidate_texts]
rerank_scores = reranker.predict(pairs)

# Sort by re-rank score
reranked = sorted(zip(candidate_ids, rerank_scores), key=lambda x: -x[1])

print("--- Re-ranked Top 5 ---")
for doc_id, score in reranked[:5]:
    idx = int(doc_id.split("_")[1])
    meta = metadatas[idx]
    print(f"Re-rank Score: {score:.4f} | File: {meta['file']}, Function: {meta['name']} (lines {meta['start_line']}-{meta['end_line']})")

--- Re-ranked Top 5 ---
Re-rank Score: 3.8826 | File: target_repo/src/flask\app.py, Function: url_for (lines 1105-1225)
Re-rank Score: 3.1221 | File: target_repo/src/flask\wrappers.py, Function: Request (lines 18-219)
Re-rank Score: 2.9628 | File: target_repo/src/flask\debughelpers.py, Function: __init__ (lines 57-78)
Re-rank Score: 2.1947 | File: target_repo/src/flask\helpers.py, Function: url_for (lines 200-251)
Re-rank Score: 0.5496 | File: target_repo/src/flask\app.py, Function: Flask (lines 110-1628)
